<a href="https://colab.research.google.com/github/fawaztanigbola/Nigeria-ForeCasting-Model/blob/main/Models_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Model Data Preparation

### Objective
Prepare the daily temperature and geographical data for two comparable LSTM forecasting models.

### 1. Missing Data Handling
- Identified grid locations with valid elevation.
- Retained locations with at least 90% complete daily minimum and maximum temperature observations.
- Excluded incomplete daily observations from sequence generation.
- Required all seven input days and the next-day target to have valid temperature values.

### 2. Common Eligible Locations
Both models use exactly the same eligible grid locations and sequence indexes. This ensures that differences in model performance are not caused by differences in their training samples.

### 3. Feature Organization

**Baseline LSTM**
- Minimum temperature (Tmin)
- Maximum temperature (Tmax)

**Geospatial LSTM**
- Minimum temperature (Tmin)
- Maximum temperature (Tmax)
- Latitude
- Longitude
- Elevation

Geographical features are static for each grid location and are repeated across the seven input time steps.

### 4. Sequence Construction
A sliding seven-day historical window is used to predict the following day's minimum and maximum temperatures.

- Input sequence length: 7 days
- Baseline input features: 2
- Geospatial input features: 5
- Prediction targets: next-day Tmin and Tmax

Invalid windows containing missing input or target temperatures are excluded.

### 5. Dataset Splitting
The data is divided chronologically:
- Training: 70%
- Validation: 15%
- Testing: 15%

The split is based on the target date, preventing future target observations from entering the training set.

### 6. Feature Normalization
Temperature normalization statistics are calculated using the training period only. Geographical features are standardized separately. The same normalization statistics are used by both models.

### 7. Memory-Efficient Sequence Loading
Rather than storing every expanded sequence in memory, the project stores cleaned temperature arrays and valid sequence indexes. A reusable PyTorch Dataset constructs sequences when requested.

### Output
The prepared dataset is saved as:

`model_data/prepared_model_data.npz`

It contains:
- Cleaned temperature observations
- Geographical features
- Eligible grid coordinates
- Training, validation and testing sequence indexes
- Normalization statistics
- Chronological split boundaries

### Important Limitation
The predictions represent eligible ERA5 grid locations, not street-level temperatures. The spatial resolution of the underlying reanalysis data limits the geographical detail of the forecasts.

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path


from google.colab import drive
drive.mount('/content/drive')

project_dir = Path(
    "/content/drive/MyDrive/Nigeria_Temperature_Project"
)

daily_file = (
    project_dir / "processed" /
    "combined_daily_temperature_2015_2024.nc"
)

ds = xr.open_dataset(daily_file)

print(ds)

Mounted at /content/drive
<xarray.Dataset> Size: 387MB
Dimensions:     (valid_time: 3653, latitude: 101, longitude: 131)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 29kB 2015-01-01 ... 2024-12-31
  * latitude    (latitude) float64 808B 14.0 13.9 13.8 13.7 ... 4.3 4.2 4.1 4.0
  * longitude   (longitude) float64 1kB 2.0 2.1 2.2 2.3 ... 14.7 14.8 14.9 15.0
    number      int64 8B ...
Data variables:
    tmin        (valid_time, latitude, longitude) float32 193MB ...
    tmax        (valid_time, latitude, longitude) float32 193MB ...
    elevation   (latitude, longitude) float32 53kB ...


Extract eligible grid locations.

In [3]:
# Extract the static elevation grid
elevation = ds["elevation"].values

# Extract the geographical coordinates
latitude = ds["latitude"].values
longitude = ds["longitude"].values

# Find locations with valid elevation
valid_elevation = np.isfinite(elevation)

# Get the row and column positions of those locations
lat_idx, lon_idx = np.where(valid_elevation)

# Extract temperature only at those locations
tmin = ds["tmin"].isel(
    latitude=xr.DataArray(lat_idx, dims="location"),
    longitude=xr.DataArray(lon_idx, dims="location")
).values

tmax = ds["tmax"].isel(
    latitude=xr.DataArray(lat_idx, dims="location"),
    longitude=xr.DataArray(lon_idx, dims="location")
).values

# Combine temperatures into one array
# Shape: (days, locations, 2)
temperature = np.stack([tmin, tmax], axis=-1).astype("float32")

# Calculate the fraction of complete days at each location
valid_days = np.isfinite(temperature).all(axis=2)
coverage = valid_days.mean(axis=0)

# Keep locations with at least 90% coverage
eligible = coverage >= 0.90

temperature = temperature[:, eligible, :]

# Keep the matching geographical features
eligible_lat = latitude[lat_idx][eligible]
eligible_lon = longitude[lon_idx][eligible]
eligible_elevation = elevation[lat_idx, lon_idx][eligible]

print("Eligible locations:", len(eligible_lat))
print("Temperature shape:", temperature.shape)

Eligible locations: 12084
Temperature shape: (3653, 12084, 2)


In [4]:
geo_features = np.column_stack([
    eligible_lat,
    eligible_lon,
    eligible_elevation
]).astype("float32")

dates = pd.to_datetime(ds["valid_time"].values)

print("Geographical features:", geo_features.shape)
print("Feature order: latitude, longitude, elevation")

Geographical features: (12084, 3)
Feature order: latitude, longitude, elevation


In [5]:
# A day is valid only when both temperatures exist
valid_day = np.isfinite(temperature).all(axis=2)

# Create rolling windows of 8 days:
# 7 input days + 1 target day
valid_windows = np.lib.stride_tricks.sliding_window_view(
    valid_day,
    window_shape=8,
    axis=0
).all(axis=-1)

# Get the time and location indexes for valid sequences
target_times, location_indexes = np.where(valid_windows)

# The target is the eighth day in each window
target_times = target_times + 7

print("Valid sequences:", len(target_times))

Valid sequences: 44058264


In [6]:
total_days = len(dates)

train_end = int(total_days * 0.70)
val_end = int(total_days * 0.85)

# Split using the date of the prediction target
train_mask = target_times < train_end

val_mask = (
    (target_times >= train_end) &
    (target_times < val_end)
)

test_mask = target_times >= val_end

# Store each split as (target time, location) pairs
train_pairs = np.column_stack([
    target_times[train_mask],
    location_indexes[train_mask]
]).astype("int32")

val_pairs = np.column_stack([
    target_times[val_mask],
    location_indexes[val_mask]
]).astype("int32")

test_pairs = np.column_stack([
    target_times[test_mask],
    location_indexes[test_mask]
]).astype("int32")

print("Training sequences:", len(train_pairs))
print("Validation sequences:", len(val_pairs))
print("Testing sequences:", len(test_pairs))

Training sequences: 30814200
Validation sequences: 6622032
Testing sequences: 6622032


In [7]:
# Temperature statistics from the training period
train_temperature = temperature[:train_end]

temp_mean = np.nanmean(
    train_temperature,
    axis=(0, 1)
)

temp_std = np.nanstd(
    train_temperature,
    axis=(0, 1)
)

# Avoid division by zero
temp_std = np.where(temp_std == 0, 1, temp_std)

# Geographical feature statistics
geo_mean = geo_features.mean(axis=0)
geo_std = geo_features.std(axis=0)

geo_std = np.where(geo_std == 0, 1, geo_std)

print("Temperature means:", temp_mean)
print("Temperature standard deviations:", temp_std)
print("Geographical means:", geo_mean)
print("Geographical standard deviations:", geo_std)

Temperature means: [17.375145 29.10062 ]
Temperature standard deviations: [5.580454 5.302514]
Geographical means: [  9.354959   8.772136 401.5107  ]
Geographical standard deviations: [  2.758269    3.6693728 289.1974   ]


In [8]:
output_dir = project_dir / "model_data"
output_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    output_dir / "prepared_model_data.npz",

    temperature=temperature,
    geo_features=geo_features,

    dates=dates.astype("datetime64[ns]"),

    eligible_lat=eligible_lat,
    eligible_lon=eligible_lon,
    eligible_elevation=eligible_elevation,

    train_pairs=train_pairs,
    val_pairs=val_pairs,
    test_pairs=test_pairs,

    temp_mean=temp_mean,
    temp_std=temp_std,

    geo_mean=geo_mean,
    geo_std=geo_std,

    train_end=train_end,
    val_end=val_end
)

print("Prepared model data saved successfully.")

Prepared model data saved successfully.


In [9]:
import torch
from torch.utils.data import Dataset

class TemperatureDataset(Dataset):

    def __init__(self, data, split="train", use_geo=False):

        self.temperature = data["temperature"]
        self.geo_features = data["geo_features"]

        self.temp_mean = data["temp_mean"]
        self.temp_std = data["temp_std"]

        self.geo_mean = data["geo_mean"]
        self.geo_std = data["geo_std"]

        self.use_geo = use_geo

        self.pairs = data[f"{split}_pairs"]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):

        target_time, location = self.pairs[index]

        # Seven days before the target day
        x_temp = self.temperature[
            target_time - 7:target_time,
            location,
            :
        ]

        # Next-day temperature targets
        y = self.temperature[target_time, location, :]

        # Standardize temperatures
        x_temp = (x_temp - self.temp_mean) / self.temp_std
        y = (y - self.temp_mean) / self.temp_std

        if self.use_geo:

            # Get and standardize static geographical features
            geo = self.geo_features[location]
            geo = (geo - self.geo_mean) / self.geo_std

            # Repeat geographical features across seven days
            geo = np.tile(geo, (7, 1))

            # Combine temperature and geographical features
            x = np.concatenate([x_temp, geo], axis=1)

        else:
            x = x_temp

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

In [10]:
# Load the prepared data
saved = np.load(
    output_dir / "prepared_model_data.npz"
)

data = {key: saved[key] for key in saved.files}

# Baseline datasets
baseline_train = TemperatureDataset(
    data, split="train", use_geo=False
)

baseline_val = TemperatureDataset(
    data, split="val", use_geo=False
)

baseline_test = TemperatureDataset(
    data, split="test", use_geo=False
)

# Geospatial datasets
geo_train = TemperatureDataset(
    data, split="train", use_geo=True
)

geo_val = TemperatureDataset(
    data, split="val", use_geo=True
)

geo_test = TemperatureDataset(
    data, split="test", use_geo=True
)

print("Baseline input shape:", baseline_train[0][0].shape)
print("Geospatial input shape:", geo_train[0][0].shape)
print("Target shape:", baseline_train[0][1].shape)

Baseline input shape: torch.Size([7, 2])
Geospatial input shape: torch.Size([7, 5])
Target shape: torch.Size([2])
